# Qwen2-VL — image and video understanding

Qwen2-VL is Alibaba's vision-language family, notable for:

- Native dynamic resolution (no forced 336/448 square crop — long receipts and dense diagrams stay readable).
- Strong multilingual OCR (CJK + Latin).
- Out-of-the-box video input via uniform frame sampling.

This notebook uses **Qwen2-VL-2B-Instruct** by default (smallest); the 7B variant works on `standard` (with int4) and `pro` (bf16). 72B is workstation-only.

Requires `transformers>=4.46` (pinned in `pip-requirements-cv.txt`).

In [ ]:
import os
PROFILE = os.environ.get('AURUM_PROFILE', 'standard')
MODEL_ID = {
    'lite':        'Qwen/Qwen2-VL-2B-Instruct',
    'standard':    'Qwen/Qwen2-VL-2B-Instruct',
    'pro':         'Qwen/Qwen2-VL-7B-Instruct',
    'workstation': 'Qwen/Qwen2-VL-7B-Instruct',
}.get(PROFILE, 'Qwen/Qwen2-VL-2B-Instruct')
print('Using:', MODEL_ID)

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
).eval()
print('Loaded on:', model.device)

In [ ]:
# Qwen2-VL accepts a chat-formatted message list with multimodal `content`
# blocks. Each block is either {type:'image', image:URL_or_path} or {type:'text', text:...}.
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': 'https://ultralytics.com/images/bus.jpg'},
            {'type': 'text',  'text':  'Describe what is happening in this photo, then count the people.'},
        ],
    },
]

# apply_chat_template handles the {image_pad} token insertion + role tags.
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Pull the actual images out of the messages list. transformers>=4.46 ships a
# `process_vision_info` helper in `qwen_vl_utils` but it's not bundled in older
# patch releases; the manual extraction below is portable.
import urllib.request
from PIL import Image
images = []
for m in messages:
    for c in m['content']:
        if c.get('type') == 'image':
            src = c['image']
            if src.startswith('http'):
                fn = '/tmp/aurum-qwen2vl-input.jpg'
                urllib.request.urlretrieve(src, fn)
                src = fn
            images.append(Image.open(src).convert('RGB'))

inputs = processor(text=[text], images=images, padding=True, return_tensors='pt').to(model.device)

In [ ]:
with torch.inference_mode():
    out_ids = model.generate(**inputs, max_new_tokens=384, do_sample=False)

# Trim the prompt prefix off each generation so we only print the assistant turn.
trimmed = [g[len(i):] for i, g in zip(inputs.input_ids, out_ids)]
for chunk in processor.batch_decode(trimmed, skip_special_tokens=True):
    print(chunk)

## Video input (advanced)

Qwen2-VL also accepts `{type: 'video', video: 'path/to/clip.mp4', max_pixels: 360*420, fps: 1.0}` blocks. The processor extracts frames at the requested FPS and stitches them into a single multi-image sequence. See the Qwen2-VL [model card](https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct) for the exact schema.